# Bermudan Swaption CVA — Deep BSDE Approach

**Reference**: She & Grecu (2018), *Neural Network for CVA: Learning Future Values*, arXiv:1811.08726, Section 3.

## Problem
Price a cash-settled **receiver Bermudan swaption** under the Hull-White 1-factor model, compute future exposures (EPE/ENE) and CVA using a neural-network BSDE solver.

## BSDE formulation
$$dX_t = [y(t) - \kappa x(t)]\,dt + \sigma_r\,dW_t \qquad \text{(HW short-rate factor)}$$
$$d\tilde{V}(t) = Z(t)\,dW_t \qquad \text{(driver = 0, }\tilde{V}\text{ is a Q-martingale)}$$
$$\tilde{V}(T_m^-) = \max\!\left(\tilde{V}(T_m^+),\; \tilde{U}(T_m)\right) \qquad \text{(early-exercise jump, eq 2.14)}$$

## Three-stage algorithm (Section 3.1)
| Stage | Function | What happens |
|---|---|---|
| 1 — Pre-training | `pre_training()` | Simulate $x(t)$ paths; compute $U(T_m)$ and $B(T_m)$ analytically |
| 2 — Training | `train()` | Picard iteration: evaluate net → apply jumps → retrain |
| 3 — Post-training | `compute_exposure()` | EPE, ENE, CVA, exercise times |

## Files
- `hw_bermudan_helpers.py` — all classes and functions (import only, no side effects)
- This notebook — runs the pipeline cell by cell


## 0. Imports

In [ ]:
import time
import logging
import numpy as np
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(message)s')

from hw_bermudan_helpers import (
    DEVICE,
    VolSurface,          # piecewise-constant sigma_r(t) — NEW
    Config,
    HullWhite,
    ValueNetwork,
    pre_training,
    train,
    compute_exposure,
    fit_bachelier,
    plot_epe_ene,
    plot_loss_curve,
    plot_future_value_evolution,
    plot_bachelier_fit,
    plot_exercise_time_distribution,
    export_to_excel,
)

import torch
torch.manual_seed(42)
np.random.seed(42)

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
print(f'Device: {DEVICE}')

## 1. Configuration

All parameters live in `Config`. Defaults match Section 3.2 of the paper.

### Volatility term structure — `sigma_r`

The paper states the HW model is **calibrated to market data on 1/18/2018**,
meaning `sigma_r(t)` is a piecewise-constant term structure bootstrapped from
swaption implied vols — not a flat constant.

`sigma_r` now accepts two types:

| Type | Example | When to use |
|---|---|---|
| `float` | `sigma_r=0.01` | Quick default, flat constant |
| `VolSurface` | `sigma_r=VolSurface(breakpoints, sigmas)` | Calibrated term structure |

`HullWhite` normalises both to a `VolSurface` internally — `y(t)`, `ln_A`,
and the Euler-Maruyama step all use the general piecewise form automatically.
Switch between the two by changing `SIGMA_R` in the cell below.


In [ ]:
# ── Volatility term structure ─────────────────────────────────────────────
# Option A: flat constant (paper default)
sigma_r_flat = 0.01

# Option B: calibrated piecewise-constant surface.
# Breakpoints align with the exercise dates + swap maturity.
# Replace these sigmas with values bootstrapped from your swaption vol surface.
sigma_r_calibrated = VolSurface(
    breakpoints = [1.5,   2.0,   2.5,   3.0,   3.5,   7.5],
    sigmas      = [0.008, 0.009, 0.010, 0.011, 0.012, 0.013],
)

# ── Select which vol to use ───────────────────────────────────────────────────
SIGMA_R = sigma_r_flat   # change to sigma_r_calibrated for the term structure

print('Selected vol surface:')
if isinstance(SIGMA_R, VolSurface):
    print(SIGMA_R)
else:
    print(f'  Flat sigma_r = {SIGMA_R}')

# ── Config ────────────────────────────────────────────────────────────────────
cfg = Config(
    # Hull-White
    kappa   = 0.01,
    sigma_r = SIGMA_R,     # accepts float OR VolSurface
    r0      = 0.028,
    # Swap
    notional   = 10_000.0,
    fixed_rate = 0.028,
    swap_tenor = 4.0,
    fixed_freq = 0.5,
    float_freq = 0.25,
    # Exercise
    exercise_start = 1.5,
    exercise_end   = 3.5,
    exercise_freq  = 0.5,
    # Simulation
    num_paths = 5_000,
    dt        = 1 / 52,
    # Network
    hidden_dim        = 11,
    num_hidden_layers = 2,
    # Training
    num_train_steps = 2_000,
    learning_rate   = 1e-3,
    lr_decay_steps  = 1_000,
    lr_decay_rate   = 0.5,
    log_every       = 500,
    # CVA
    hazard_rate = 0.01,
    recovery    = 0.40,
)

hw = HullWhite(cfg)

print(f'\nExercise dates : {cfg.exercise_dates}')
print(f'Horizon T      : {cfg.T:.1f}y')
print(f'Time steps N   : {cfg.N}  (dt={cfg.dt:.4f})')
print(f'Paths          : {cfg.num_paths:,}')
print(f'Train steps    : {cfg.num_train_steps:,}')

## 2. Stage 1 — Pre-Training

Simulate $x(t)$ paths via **Euler-Maruyama** and precompute, at each exercise date $T_m$:
- $U(T_m) = \max(V_{\text{receiver\ swap}}, 0)$ — undiscounted exercise value (analytical)
- $B(T_m) = \exp\!\left(\int_0^{T_m} r(s)\,ds\right)$ — numeraire (trapezoidal rule on stored paths)
- $\tilde{U}(T_m) = U(T_m) / B(T_m)$ — discounted exercise value used in the jump condition

No neural network is involved here.

In [ ]:
t0 = time.time()
pre_data = pre_training(cfg, hw)
print(f'\nStage 1 done in {time.time()-t0:.1f}s')
print(f"X_paths shape : {pre_data['X_paths'].shape}")
print(f"U_exercise    : {pre_data['U_exercise'].shape}")
print(f"U_tilde       : {pre_data['U_tilde'].shape}")

### Stage 1 sanity check — sample x(t) paths and exercise values
A few simulated short-rate factor paths plus the distribution of exercise values at each date.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: sample x(t) paths
t_grid = pre_data['time_grid']
for p in range(min(30, cfg.num_paths)):
    axes[0].plot(t_grid, pre_data['X_paths'][p], alpha=0.4, linewidth=0.7)
for T_m in cfg.exercise_dates:
    axes[0].axvline(T_m, color='red', linestyle=':', alpha=0.6, linewidth=0.8)
axes[0].set_xlabel('Time (years)', fontsize=11)
axes[0].set_ylabel('x(t)', fontsize=11)
axes[0].set_title('Sample HW Short-Rate Factor Paths\n(red dashes = exercise dates)', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Right: box-plot of exercise values at each T_m
ex_vals = [pre_data['U_exercise'][:, m] for m in range(len(cfg.exercise_dates))]
axes[1].boxplot(ex_vals, labels=[f'T={T:.1f}' for T in cfg.exercise_dates])
axes[1].set_xlabel('Exercise Date', fontsize=11)
axes[1].set_ylabel('Exercise Value U(T_m)', fontsize=11)
axes[1].set_title('Distribution of Exercise Values by Date', fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Stage 2 — Training (Picard Iteration with Exercise Jumps)

Each Picard iteration does:
1. **Evaluate** `Y_net(t, x)` on all paths → $\tilde{V}$ raw
2. **Apply exercise jumps**: $\tilde{V}(T_m) \leftarrow \max(\tilde{V}(T_m),\; \tilde{U}(T_m))$
3. **Build targets**: since the driver is zero, `target(t_i) = Ṽ_jumped(t_{i+1})`
4. **Retrain** `Y_net` via Adam to minimise MSE against those targets

The jump in step 2 is the key Bermudan feature — it enforces optimal stopping in the training targets.

In [ ]:
NUM_PICARD_ITERS = 3   # increase for more refinement

t0 = time.time()
net, all_losses, Y_snapshots = train(
    cfg, hw, pre_data, num_picard_iters=NUM_PICARD_ITERS
)
print(f'\nStage 2 done in {time.time()-t0:.1f}s')
print(f'Total loss entries : {len(all_losses)}')
print(f'Y snapshots stored : {len(Y_snapshots)}')

## 4. Stage 3 — Exposure & CVA

Re-evaluates the trained network on all paths with exercise jumps, then:
- Determines exercise indicators $\eta_{pm}$ at each $T_m$
- Converts discounted $\tilde{V}$ back to undiscounted $V = \tilde{V} \cdot B$
- Zeros out exposure on exercised paths (cash-settled, eq 2.18)
- Computes EPE, ENE, CVA, and exercise-time artifacts $\tau_p$

In [ ]:
t0 = time.time()
exp_data = compute_exposure(cfg, hw, net, pre_data)
print(f'\nStage 3 done in {time.time()-t0:.1f}s')
print(f"\nCVA             = {exp_data['CVA']:.6f}")
print(f"EPE at t=0      = {exp_data['EPE'][0]:.4f}")
frac_ex = np.mean(~np.isnan(exp_data['exercise_times']))
print(f"% paths exercised = {frac_ex:.1%}")

## 5. Figure 3.1 — EPE/ENE Profiles and Loss Curve

**Left (Fig 3.1 left)**: EPE decreases over time and drops at each exercise date (paths that exercise contribute zero exposure thereafter). ENE ≈ 0 since the receiver swaption payoff is always non-negative.

**Right (Fig 3.1 right)**: Training loss should plateau around step 500 (paper).

In [ ]:
fig_epe  = plot_epe_ene(exp_data, cfg)
fig_loss = plot_loss_curve(all_losses, cfg)
plt.show()

## 6. Figure 3.2 — Future Value Evolution at 1st Exercise Date

Scatter of $(x, V)$ at $T_1 = 1.5\text{y}$ at each Picard iteration snapshot.
- **Blue**: portfolio value $V$ (learned — starts noisy, converges to a smooth option-price curve)
- **Orange**: exercise value $U$ (fixed, analytical — linear in $x$)

At convergence $V$ should interpolate between 0 (deep OTM) and $U$ (deep ITM).

In [ ]:
fig_fv1 = plot_future_value_evolution(
    exp_data, pre_data, Y_snapshots,
    exercise_date_idx=0,   # 0 = first exercise date (T=1.5y)
    cfg=cfg,
)
plt.show()

## 7. Figure 3.3 — Future Value Evolution at 4th Exercise Date

Same plot as Fig 3.2 but at $T_4 = 3.0\text{y}$. The deeper ITM region is typically larger here because more rate moves have accumulated, making the swap more likely to be in the money.

In [ ]:
if len(cfg.exercise_dates) >= 4:
    fig_fv4 = plot_future_value_evolution(
        exp_data, pre_data, Y_snapshots,
        exercise_date_idx=3,   # 3 = fourth exercise date (T=3.0y)
        cfg=cfg,
    )
    plt.show()
else:
    print('Fewer than 4 exercise dates configured — skipping Fig 3.3')

## 8. Figure 3.4 — Bachelier Fit

Fits the learned $V(x)$ at the first exercise date to the Bachelier-type formula (eq 3.8):
$$V_{\text{Bach}}(x) = A\left[(x-c)\,\Phi\!\left(\frac{x-c}{s}\right) + s\,\phi\!\left(\frac{x-c}{s}\right)\right]$$

- $A$ ≈ slope of exercise value (DV01-like)
- $c$ ≈ ATM strike in $x$-space
- $s$ ≈ effective normal volatility (controls ATM rounding)

In [ ]:
for ex_idx in range(min(2, len(cfg.exercise_dates))):
    fig_bach = plot_bachelier_fit(exp_data, pre_data, exercise_date_idx=ex_idx, cfg=cfg)
    plt.show()

## 9. Exercise Time Artifacts

Three views of the exercise time distribution:
- **Histogram** of $\tau_p$ (the date each path exercises, NaN = unexercised)
- **Exercise probability** at each $T_m$ (fraction of alive paths that exercise)
- **Survival curve** $P(\text{not yet exercised by } T_m)$

In [ ]:
fig_ex = plot_exercise_time_distribution(exp_data, cfg)
plt.show()

import pandas as pd
df_ex = pd.DataFrame({
    'Exercise Date': cfg.exercise_dates,
    'Exercise Prob': exp_data['exercise_prob'],
    'Survival Prob': exp_data['survival_curve'],
}).set_index('Exercise Date')
display(df_ex.round(4))

## 10. CVA Summary

$$\text{CVA} = (1-R)\sum_n \text{EPE}(T_n)\,\Delta\text{PD}(T_n)$$
with flat hazard rate $\lambda$ and $\Delta\text{PD}(T_n) = e^{-\lambda T_{n-1}} - e^{-\lambda T_n}$.

In [ ]:
print('=' * 45)
print('CVA SUMMARY')
print('=' * 45)
print(f"CVA                  = {exp_data['CVA']:.6f}")
print(f"Hazard rate (λ)      = {cfg.hazard_rate:.2%}")
print(f"Recovery rate (R)    = {cfg.recovery:.0%}")
print(f"EPE at t=0           = {exp_data['EPE'][0]:.4f}")
print(f"Max EPE              = {exp_data['EPE'].max():.4f}")
print(f"ENE range            = [{exp_data['ENE'].min():.4f}, {exp_data['ENE'].max():.4f}]")
frac = np.mean(~np.isnan(exp_data['exercise_times']))
print(f"Paths exercised      = {frac:.1%}")
print(f"Final train loss     = {all_losses[-1]:.4e}")
print('=' * 45)

## 11. Export Results to Excel

Writes a formatted 5-sheet workbook:
| Sheet | Contents |
|---|---|
| Summary | Parameters + headline results |
| Exposure Profiles | EPE, ENE, discount factor at every time step |
| Exercise Times | $\tau_p$ per path + exercised flag |
| Exercise Analytics | Probability and survival by date |
| Training Loss | MSE loss at every gradient step |

In [ ]:
OUTPUT_PATH = 'bermudan_swaption_results.xlsx'

export_to_excel(
    cfg        = cfg,
    pre_train_data = pre_data,
    exposure_data  = exp_data,
    all_losses     = all_losses,
    output_path    = OUTPUT_PATH,
)
print(f'Saved: {OUTPUT_PATH}')

## 12. Optional: Sensitivity to σ_r

Re-run the full pipeline for a higher volatility to see the effect on EPE shape and CVA. *(This cell is optional — it trains a second network from scratch.)*

In [ ]:
# Optional: compare flat sigma_r vs calibrated VolSurface side-by-side.
# Uncomment to run — trains a second network from scratch.

# cfg_cal = Config(
#     sigma_r = sigma_r_calibrated,
#     num_train_steps = 2_000, log_every = 500,
# )
# hw_cal       = HullWhite(cfg_cal)
# pre_cal      = pre_training(cfg_cal, hw_cal)
# net_cal, losses_cal, _ = train(cfg_cal, hw_cal, pre_cal, num_picard_iters=3)
# exp_cal      = compute_exposure(cfg_cal, hw_cal, net_cal, pre_cal)
#
# fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# t = exp_data['time_grid']
# axes[0].plot(t, exp_data['EPE'], label='EPE  flat')
# axes[0].plot(t, exp_cal['EPE'],  label='EPE  calibrated', linestyle='--')
# axes[0].set_title('EPE comparison'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
# axes[1].plot(t, exp_data['ENE'], label='ENE  flat')
# axes[1].plot(t, exp_cal['ENE'],  label='ENE  calibrated', linestyle='--')
# axes[1].set_title('ENE comparison'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
# plt.tight_layout(); plt.show()
# print(f'CVA flat       = {exp_data["CVA"]:.4f}')
# print(f'CVA calibrated = {exp_cal["CVA"]:.4f}')
print('Sensitivity cell is commented out — uncomment to run.')